# Vietnamese Dependency Parser — Training on Colab

**Trước khi chạy:** Vào `Runtime > Change runtime type > T4 GPU`

In [ ]:
# Kiểm tra GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU - vào Runtime > Change runtime type > T4 GPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')

## 1. Mount Google Drive & Upload project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/dep-parsing/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/dep-parsing/runs', exist_ok=True)
print('Drive mounted OK')

In [ ]:
# Upload project từ máy tính lên Colab
# Nén folder dep-parsing thành zip rồi chạy cell này
from google.colab import files
uploaded = files.upload()  # chọn file dep-parsing.zip

In [ ]:
import zipfile, os

with zipfile.ZipFile('dep-parsing.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content/dep-parsing')
print('Working dir:', os.getcwd())
!ls

## 2. Cài dependencies

In [ ]:
!pip install -q transformers==4.40.0 tokenizers sentencepiece tqdm
print('Dependencies installed')

## 3. Visualization — chọn TensorBoard hoặc W&B

Chạy **một trong hai** cell bên dưới.

In [ ]:
# ── OPTION A: TensorBoard (không cần tài khoản) ──────────────────────────────
# Chạy cell này TRƯỚC khi train. TensorBoard sẽ tự cập nhật theo thời gian thực.
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/dep-parsing/runs

In [ ]:
# ── OPTION B: Weights & Biases (UI đẹp hơn, lưu cloud vĩnh viễn) ─────────────
# Tạo tài khoản miễn phí tại wandb.ai, lấy API key trong Settings > API keys
!pip install -q wandb
import wandb
wandb.login()  # nhập API key khi được hỏi

## 4. Chuẩn bị dữ liệu

In [2]:
!python prepare_data.py
!echo '---'
!wc -l prepared_data/dp2020/train.conllu prepared_data/dp2020/dev.conllu

Saved total gold (1123 sentences) to prepared_data/dp2020/total-gold.conllu
Saved test gold input to prepared_data/dp2020/test_gold_input.conllu
Saved train split to prepared_data/dp2020/train.conllu
Saved dev split to prepared_data/dp2020/dev.conllu
Saved summary to prepared_data/dp2020/README.txt
---
 149550 prepared_data/dp2020/train.conllu
  16725 prepared_data/dp2020/dev.conllu
 166275 total


## 5. Huấn luyện

Chọn **một** cấu hình bên dưới. Flag `--log-dir` dùng cho TensorBoard, `--wandb` dùng cho W&B.

In [ ]:
# ============================================================
# CẤU HÌNH A: Nhanh — PhoBERT + MST + tách MLP
# ~30–45 phút trên T4
# ============================================================
!python train.py \
    --train-path prepared_data/dp2020/train.conllu \
    --dev-path   prepared_data/dp2020/dev.conllu \
    --use-mst \
    --arc-hidden-dim 512 --label-hidden-dim 128 \
    --bert-lr 2e-5 --head-lr 1e-3 \
    --batch-size 32 --epochs 15 --patience 5 \
    --log-dir /content/drive/MyDrive/dep-parsing/runs/config-a \
    --save-path /content/drive/MyDrive/dep-parsing/checkpoints/fast.pt
    # thêm --wandb nếu dùng W&B thay TensorBoard

In [ ]:
# ============================================================
# CẤU HÌNH B: Đầy đủ — PhoBERT + MST + POS + Char CNN
# ~60–90 phút trên T4
# ============================================================
!python train.py \
    --train-path prepared_data/dp2020/train.conllu \
    --dev-path   prepared_data/dp2020/dev.conllu \
    --use-mst \
    --pos-dim 50 \
    --char-out-dim 50 --char-embed-dim 50 --max-word-len 30 \
    --arc-hidden-dim 512 --label-hidden-dim 128 \
    --bert-lr 2e-5 --head-lr 1e-3 \
    --batch-size 32 --epochs 20 --patience 5 \
    --log-dir /content/drive/MyDrive/dep-parsing/runs/config-b \
    --save-path /content/drive/MyDrive/dep-parsing/checkpoints/full.pt
    # thêm --wandb nếu dùng W&B thay TensorBoard

In [ ]:
# ============================================================
# CẤU HÌNH C: Test nhanh code (1 epoch)
# ~3–5 phút
# ============================================================
!python train.py \
    --train-path prepared_data/dp2020/train.conllu \
    --dev-path   prepared_data/dp2020/dev.conllu \
    --batch-size 16 --epochs 1 \
    --log-dir /content/runs/test \
    --save-path /content/checkpoints/test.pt

## 6. Đánh giá trên test set

In [ ]:
CHECKPOINT = '/content/drive/MyDrive/dep-parsing/checkpoints/full.pt'

!python predict.py \
    --checkpoint {CHECKPOINT} \
    --input-file prepared_data/dp2020/test_input.conllu \
    --output-file /content/predictions.conllu

print('Predictions saved.')

In [ ]:
gold_file = 'prepared_data/dp2020/total-gold.conllu'

!python evaluate.py \
    --gold-path {gold_file} \
    --pred-path /content/predictions.conllu

## 7. Download checkpoint về máy (tuỳ chọn)

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/dep-parsing/checkpoints/full.pt')